In [ ]:
from pygooglenews import GoogleNews
import pandas as pd
import time
from datetime import datetime
from dateutil import parser

# Inicializa
gn = GoogleNews(lang='pt', country='BR')

# 1. Dicionário de Empresas com LISTA de palavras-chave
# A primeira palavra da lista é usada como identificador principal
config_empresas = {
    'PETR4': ['Petrobras', 'PETR4', 'PETR3'],
    'ITUB4': ['Itaú Unibanco', 'ITUB4', 'Banco Itaú'],
    'VALE3': ['Vale S.A.', 'VALE3', 'Vale mineradora'], # "Vale" sozinho foi removido
    'BPAC11': ['BTG Pactual', 'BPAC11', 'Banco BTG'],
    'ABEV3': ['Ambev', 'ABEV3'],
    'BBDC4': ['Bradesco', 'BBDC4', 'Banco Bradesco'],
    'WEGE3': ['WEG S.A.', 'WEGE3', 'WEG Motores', 'Empresa WEG'], # Evita "weg" solto
    'ITSA4': ['Itaúsa', 'ITSA4'],
    'BBAS3': ['Banco do Brasil', 'BBAS3'],
    'ELET3': ['Eletrobras', 'ELET3', 'ELET6']
}

meses = pd.date_range(start='2023-01-01', end='2024-12-31', freq='MS')

lista_final = []

print("--- Iniciando Coleta Multi-Keyword ---")

for ticker, keywords in config_empresas.items():
    print(f"Coletando: {ticker} | Keywords: {keywords}")

    # Monta a string de busca com OR
    # Ex: (intitle:"Vale S.A." OR intitle:"VALE3")
    query_keywords = " OR ".join([f'intitle:"{k}"' for k in keywords])
    base_query = f"({query_keywords})"

    for mes in meses:
        data_ini = mes.strftime('%Y-%m-%d')
        data_fim = (mes + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

        # Query completa com data
        query = f'{base_query} after:{data_ini} before:{data_fim}'

        try:
            search = gn.search(query)
            entries = search['entries']

            for item in entries:
                lista_final.append({
                    'ticker': ticker,
                    'search_term': str(keywords), # Salva quais termos usamos
                    'title': item.title,
                    'link': item.link,
                    'published': item.published,
                    'source': item.source.title
                })

            time.sleep(1) # Respeitando o servidor

        except Exception as e:
            print(f"Erro em {ticker} - {mes}: {e}")

    print(f"  > Acumulado parcial: {len(lista_final)}")

# Tratamento e Salvamento
if lista_final:
    df = pd.DataFrame(lista_final)

    # Converte data
    df['datetime'] = df['published'].apply(lambda x: parser.parse(x).strftime('%Y-%m-%d'))

    # Remove duplicatas (Importantíssimo pois 'Vale S.A.' e 'VALE3' podem trazer a mesma notícia)
    df = df.drop_duplicates(subset=['title', 'ticker'])

    # Salva
    arquivo_nome = 'dataset_10_gigantes_tcc.csv'
    df.to_csv(arquivo_nome, index=False)
    print(f"\n✅ SUCESSO! Arquivo salvo: {arquivo_nome} com {len(df)} notícias únicas.")
    print(df[['datetime', 'ticker', 'title']].head())
else:
    print("❌ Nenhuma notícia encontrada.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

# --- Configuração da Fonte (Arial 10) ---
try:
    # Configura a fonte para Arial (ou similar sem serifa no ambiente Colab)
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans']
    plt.rcParams['font.size'] = 10
    print("Fonte configurada para Arial (ou similar) tamanho 10.")
except Exception as e:
    print(f"Erro ao configurar fonte Arial, usando fallback: {e}")
    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.rcParams['font.size'] = 10

# --- Caminho do CSV (Conforme especificado pelo usuário) ---
CSV_PATH = '/content/sample_data/dataset_10_gigantes_tcc.csv'

# 1. Carregar e Preparar
try:
    df = pd.read_csv(CSV_PATH)
    df['datetime'] = pd.to_datetime(df['datetime'])
except FileNotFoundError:
    print(f"ERRO CRÍTICO: Arquivo não encontrado em {CSV_PATH}. Por favor, verifique o caminho e execute novamente.")
    # Interrompe a execução se o arquivo não for encontrado
    raise

# --- Função Auxiliar para Gerar a Tabela no Padrão UFV ---
def plot_tabela_ufv(data_frame, tabela_num, title, filename):

    # 1. Configurações da Figura e Eixos
    # Ajusta altura da figura baseada no número de linhas (0.3 polegadas por linha + margem)
    fig, ax = plt.subplots(figsize=(8, (len(data_frame) * 0.3) + 1.2))
    ax.axis('tight')
    ax.axis('off')

    # 2. Cria a Tabela
    cell_data = data_frame.values.astype(str) # Garante que todos os dados são strings

    table = ax.table(
        cellText=cell_data,
        colLabels=data_frame.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )

    # 3. Estilização (Arial 10, esticamento de linha)
    table.auto_set_font_size(False)
    table.set_fontsize(plt.rcParams['font.size']) # Tamanho 10
    table.scale(1, 1.2) # Estica linhas

    # 4. Configuração de Bordas (Tabela: Abertas Lateralmente, Traço Superior e Traço no Cabeçalho)
    n_rows = len(data_frame.index) + 1 # +1 para o cabeçalho
    n_cols = len(data_frame.columns)

    # Remove TODAS as bordas (Configuração inicial) e define fundo branco
    for i in range(n_rows):
        for j in range(n_cols):
            cell = table[(i, j)]
            cell.set_edgecolor('w') # Borda branca (invisível)
            cell.set_facecolor('w') # Fundo branco
            cell.set_linewidth(0.0)

    # Adiciona as bordas horizontais (Traço Superior e Traço Abaixo do Cabeçalho)
    for j in range(n_cols):
        # 1. Linha Superior da Tabela/Cabeçalho
        table[(0, j)].set_linestyle('solid')
        table[(0, j)].set_linewidth(0.5)
        table[(0, j)].set_edgecolor('k')

        # 2. Linha Inferior do Cabeçalho (divisão entre títulos e dados)
        cell_below = table[(1, j)]
        cell_below.set_linestyle('solid')
        cell_below.set_linewidth(0.5)
        cell_below.set_edgecolor('k')

    # 5. Adiciona o Título (Parte superior, fora da tabela, alinhado à esquerda)
    # y=1.02 coloca o título acima da figura
    full_title = f"Tabela {tabela_num}: {title}"
    ax.set_title(full_title, loc='left', fontsize=plt.rcParams['font.size'], y=1.02)

    # 6. Salvar Imagem
    plt.savefig(filename, dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig) # Fecha a figura para não ocupar memória
    print(f"Gerada imagem da Tabela {tabela_num}: {filename}")


# --- Geração dos Dados e Execução das Tabelas ---

# 1. Tabela 3.1: Palavras-chave, Volume de Notícias e Participação na Amostra
config_empresas = {
    'PETR4': ['Petrobras', 'PETR4', 'PETR3'],
    'ITUB4': ['Itaú Unibanco', 'ITUB4', 'Banco Itaú'],
    'VALE3': ['Vale S.A.', 'VALE3', 'Vale mineradora'],
    'BPAC11': ['BTG Pactual', 'BPAC11', 'Banco BTG'],
    'ABEV3': ['Ambev', 'ABEV3'],
    'BBDC4': ['Bradesco', 'BBDC4', 'Banco Bradesco'],
    'WEGE3': ['WEG S.A.', 'WEGE3', 'WEG Motores', 'Empresa WEG'],
    'ITSA4': ['Itaúsa', 'ITSA4'],
    'BBAS3': ['Banco do Brasil', 'BBAS3'],
    'ELET3': ['Eletrobras', 'ELET3', 'ELET6']
}
volume_por_empresa = df['ticker'].value_counts().reset_index()
volume_por_empresa.columns = ['Ticker', 'Volume Total']
volume_por_empresa['Palavras-chave'] = volume_por_empresa['Ticker'].apply(lambda t: ", ".join(config_empresas.get(t, ['N/A'])))
volume_por_empresa = volume_por_empresa[['Ticker', 'Palavras-chave', 'Volume Total']]
volume_por_empresa['% do Total'] = (volume_por_empresa['Volume Total'] / volume_por_empresa['Volume Total'].sum() * 100).round(2)

plot_tabela_ufv(volume_por_empresa, 3.1, "Palavras-chave, Volume de Notícias e Participação na Amostra", "tabela_3_1_palavras_chave.png")


# 2. Tabela 3.2: Distribuição Temporal (Volume Mensal)
df['mes_ano'] = df['datetime'].dt.to_period('M')
volume_mensal = df.groupby('mes_ano').size().reset_index()
volume_mensal.columns = ['Mês/Ano', 'Contagem']
volume_mensal['Ano'] = volume_mensal['Mês/Ano'].astype(str).str[:4]
volume_mensal['Mês'] = volume_mensal['Mês/Ano'].astype(str).str[5:]
volume_mensal = volume_mensal.drop(columns=['Mês/Ano']).reset_index(drop=True)

media_mensal = int(round(volume_mensal['Contagem'].mean()))
std_mensal = int(round(volume_mensal['Contagem'].std()))
title_3_2 = f"Distribuição Temporal do Volume de Notícias (Média: {media_mensal}; Desvio Padrão: {std_mensal})"

plot_tabela_ufv(volume_mensal, 3.2, title_3_2, "tabela_3_2_volume_mensal.png")


# 3. Tabela 3.3: Top 10 Fontes de Notícias Mais Frequentes
top_sources = df['source'].value_counts().head(10).reset_index()
top_sources.columns = ['Fonte', 'Contagem']
top_sources['Participação (%)'] = (top_sources['Contagem'] / top_sources['Contagem'].sum() * 100).round(2)

plot_tabela_ufv(top_sources, 3.3, "Top 10 Fontes de Notícias Mais Frequentes (Veículos de Mídia)", "tabela_3_3_top_fontes.png")

print("\n----------------------------------------------------------------")
print("✅ Todas as imagens das Tabelas foram geradas no formato UFV (Arial 10, bordas laterais abertas) na pasta de arquivos do Colab.")
print("Você pode baixá-las clicando no ícone da pasta e depois no ícone de download ao lado de cada arquivo.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Carregar e Preparar
df = pd.read_csv('dataset_10_gigantes_tcc.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

# Configuração de estilo para gráficos acadêmicos
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.figsize': (10, 6)})

print("=== RESUMO ESTATÍSTICO DO DATASET ===")
print(f"Total de Notícias: {len(df)}")
print(f"Período: {df['datetime'].min().date()} até {df['datetime'].max().date()}")
print("-" * 30)

# 2. Distribuição por Empresa
noticias_por_empresa = df['ticker'].value_counts()
print("\nTop 5 Empresas com mais notícias:")
print(noticias_por_empresa.head())

plt.figure(figsize=(10, 5))
ax = sns.barplot(x=noticias_por_empresa.index, y=noticias_por_empresa.values, palette="viridis")
plt.title("Distribuição do Volume de Notícias por Empresa (2023-2024)")
plt.ylabel("Quantidade de Notícias")
plt.xlabel("Ticker")
plt.xticks(rotation=45)
for i in ax.containers:
    ax.bar_label(i,)
plt.tight_layout()
plt.savefig("grafico_distribuicao_empresas.png", dpi=300) # Salva para o TCC
plt.show()

# 3. Distribuição Temporal (Volume Mensal)
# Agrupa por mês e conta
df['mes_ano'] = df['datetime'].dt.to_period('M')
volume_mensal = df.groupby('mes_ano').size()

plt.figure(figsize=(12, 5))
volume_mensal.plot(kind='line', marker='o', color='darkblue')
plt.title("Evolução Temporal da Coleta de Notícias (Mensal)")
plt.ylabel("Volume de Notícias")
plt.xlabel("Mês/Ano")
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.savefig("grafico_evolucao_temporal.png", dpi=300)
plt.show()

# 4. Principais Fontes (Sources)
top_sources = df['source'].value_counts().head(10)
print("\nTop 10 Veículos de Mídia:")
print(top_sources)

plt.figure(figsize=(10, 6))
sns.barplot(y=top_sources.index, x=top_sources.values, palette="Blues_r")
plt.title("Top 10 Fontes de Notícias Mais Frequentes")
plt.xlabel("Contagem")
plt.tight_layout()
plt.savefig("grafico_top_fontes.png", dpi=300)
plt.show()

# 5. Check de 'Buracos' nos dados
# Cria uma tabela dinâmica (Pivot Table) para ver se falta mês para alguma empresa
pivot = df.groupby([df['datetime'].dt.to_period('M'), 'ticker']).size().unstack(fill_value=0)
print("\n--- Verificação de Consistência (Amostra dos últimos 5 meses) ---")
print(pivot.tail())

In [ ]:
!pip install openai pandas tqdm nest_asyncio -q

In [ ]:
import os
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "")

In [ ]:
import asyncio, json, re
from pathlib import Path
import pandas as pd
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
nest_asyncio.apply()

# ── Configurações ──────────────────────────────────────────────
API_KEY     = os.environ["GROQ_API_KEY"]
MODEL       = "llama-3.1-8b-instant"
BATCH_SIZE  = 15
CONCURRENCY = 3
MAX_RETRIES = 6
CHECKPOINT  = "sentimentos.jsonl"
INPUT_CSV   = "dataset_10_gigantes_tcc.csv"
# ───────────────────────────────────────────────────────────────

client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=API_KEY
)

SYSTEM = "Você é um analista financeiro experiente especializado em mercado brasileiro (B3)."

def build_prompt(batch):
    linhas = "\n".join(
        f'Manchete {i+1} (empresa: {r["ticker"]}): {r["title"]}'
        for i, r in enumerate(batch)
    )
    return (
        f"Abaixo estão {len(batch)} manchetes sobre empresas da B3.\n"
        "Para cada uma, responda APENAS com POSITIVO, NEGATIVO ou NEUTRO "
        "conforme o impacto esperado no preço da ação no curto prazo.\n"
        "Responda SOMENTE com um array JSON na mesma ordem. "
        'Exemplo: ["POSITIVO","NEUTRO","NEGATIVO"]\n\n'
        + linhas
    )

def parse_labels(raw, n):
    m = re.search(r"\[.*?\]", raw, re.DOTALL)
    if not m:
        raise ValueError("Array não encontrado")
    labels = [str(l).strip().upper() for l in json.loads(m.group(0))]
    if len(labels) != n:
        raise ValueError(f"Esperado {n}, recebido {len(labels)}")
    if any(l not in {"POSITIVO","NEGATIVO","NEUTRO"} for l in labels):
        raise ValueError(f"Label inválido: {labels}")
    return labels

def load_done(path):
    done = set()
    if not Path(path).exists():
        return done
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get("sentimento") != "ERRO":
                    done.add(int(r["idx"]))
            except:
                pass
    return done

async def score_batch(indexed_batch, sem, f, lock):
    indices, rows = zip(*indexed_batch)
    rows = list(rows)
    async with sem:
        for attempt in range(MAX_RETRIES):
            try:
                resp = await client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM},
                        {"role": "user",   "content": build_prompt(rows)}
                    ],
                    temperature=0,
                    max_tokens=250,
                )
                labels = parse_labels(resp.choices[0].message.content, len(rows))
                async with lock:
                    for idx, row, label in zip(indices, rows, labels):
                        f.write(json.dumps({
                            "idx": idx, "ticker": row["ticker"],
                            "datetime": row["datetime"],
                            "title": row["title"], "sentimento": label
                        }, ensure_ascii=False) + "\n")
                    f.flush()
                return
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    await asyncio.sleep(min(4**attempt, 120))
        async with lock:
            for idx, row in zip(indices, rows):
                f.write(json.dumps({
                    "idx": idx, "ticker": row["ticker"],
                    "datetime": row["datetime"],
                    "title": row["title"], "sentimento": "ERRO"
                }, ensure_ascii=False) + "\n")
            f.flush()

async def main():
    df = pd.read_csv(INPUT_CSV).reset_index().rename(columns={"index": "idx"})
    done = load_done(CHECKPOINT)
    todo = df[~df["idx"].isin(done)]
    print(f"Total: {len(df)} | Já feitos: {len(done)} | Restantes: {len(todo)}")

    records = todo.to_dict("records")
    batches = [
        [(r["idx"], r) for r in records[i:i+BATCH_SIZE]]
        for i in range(0, len(records), BATCH_SIZE)
    ]
    print(f"{len(batches)} batches de {BATCH_SIZE} | ~{len(batches)//5}-{len(batches)//2} min estimados\n")

    sem  = asyncio.Semaphore(CONCURRENCY)
    lock = asyncio.Lock()
    mode = "a" if Path(CHECKPOINT).exists() else "w"
    with open(CHECKPOINT, mode, encoding="utf-8") as f:
        await tqdm_asyncio.gather(
            *[score_batch(b, sem, f, lock) for b in batches],
            desc="Classificando"
        )

    done_final = load_done(CHECKPOINT)
    print(f"\n✅ Concluído: {len(done_final)}/{len(df)} classificados")

await main()

In [ ]:
import json
import pandas as pd

with open("sentimentos.jsonl", encoding="utf-8") as f:
    recs = [json.loads(line) for line in f]

df = pd.DataFrame(recs)
# mantém só a última ocorrência de cada idx
df = df.drop_duplicates(subset="idx", keep="last")

n_erro = (df["sentimento"] == "ERRO").sum()
print(f"Total únicos: {len(df)}")
print(f"Erros: {n_erro} ({n_erro/len(df)*100:.1f}%)")
print(df["sentimento"].value_counts())

# remove os erros definitivamente — não entram no event study
df_final = df[df["sentimento"] != "ERRO"]
print(f"\nDataset final para análise: {len(df_final)} notícias")

# salva limpo
df_final.to_json("sentimentos.jsonl", orient="records", lines=True, force_ascii=False)

from google.colab import files
files.download("sentimentos.jsonl")

In [ ]:
!pip install yfinance scipy statsmodels requests matplotlib -q

In [ ]:
"""
SCRIPT 2: Estudo de Eventos – Pipeline Completo
===============================================
Uso (após 1_score_sentimentos.py ter terminado):
    pip install pandas yfinance scipy statsmodels requests matplotlib seaborn tqdm
    python 2_event_study.py

Requer: sentimentos.jsonl (gerado pelo script 1)
        dataset_10_gigantes_tcc.csv (base de notícias original)
Output: event_study_results.csv, tabelas no terminal, gráficos .png
"""

import json
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm
import yfinance as yf
from scipy import stats
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ─── PARÂMETROS (conforme metodologia do TCC) ─────────────────────────────────
ESTIMATION_START = -120   # dias úteis antes de T0 (início da janela de estimação)
ESTIMATION_END   = -31    # dias úteis antes de T0 (fim da janela de estimação)
# Janela de evento: T0 e T+1 → CAR[0,+1]
MIN_OBS = 30              # mínimo de observações na janela de estimação
# ──────────────────────────────────────────────────────────────────────────────

TICKERS = [
    "PETR4.SA", "BBDC4.SA", "BBAS3.SA", "ABEV3.SA",
    "ELET3.SA", "BPAC11.SA", "ITUB4.SA", "VALE3.SA",
    "WEGE3.SA", "ITSA4.SA",
]
IBOV = "^BVSP"
# Mapeamento ticker do CSV → ticker yfinance
TICKER_MAP = {t.replace(".SA", ""): t for t in TICKERS}

# ─── 1. CARREGAR SENTIMENTOS ───────────────────────────────────────────────────
def load_sentimentos(path="sentimentos.jsonl") -> pd.DataFrame:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get("sentimento") not in ("ERRO", None):
                    records.append(r)
            except Exception:
                pass
    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["datetime"])
    df["ticker_yf"] = df["ticker"].map(TICKER_MAP)
    df = df.dropna(subset=["ticker_yf"])
    mapping = {"POSITIVO": 1, "NEUTRO": 0, "NEGATIVO": -1}
    df["sentimento_num"] = df["sentimento"].map(mapping)
    return df

# ─── 2. DOWNLOAD DE PREÇOS (yfinance) ─────────────────────────────────────────
def download_prices() -> pd.DataFrame:
    print("📥 Baixando preços (yfinance)...")
    raw = yf.download(
        TICKERS + [IBOV],
        start="2022-06-01",
        end="2025-01-31",
        auto_adjust=True,
        progress=True,
    )
    # yfinance pode retornar MultiIndex ou Index simples dependendo da versão
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw["Close"]
    else:
        prices = raw[["Close"]] if "Close" in raw.columns else raw
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    prices = prices.dropna(how="all")
    print(f"   Preços: {prices.shape[0]} dias úteis, {prices.shape[1]} ativos")
    return prices

# ─── 3. DOWNLOAD SELIC/CDI (BCB SGS) ─────────────────────────────────────────
def download_selic() -> pd.Series:
    """Taxa CDI diária (série 11 do BCB), já em decimal (ex: 0.0004)."""
    print("📥 Baixando Taxa CDI/SELIC (BCB)...")
    url = (
        "https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados"
        "?formato=json&dataInicial=01/06/2022&dataFinal=31/01/2025"
    )
    try:
        r = requests.get(url, timeout=30)
        data = r.json()
        df = pd.DataFrame(data)
        df["data"]  = pd.to_datetime(df["data"], format="%d/%m/%Y")
        df["valor"] = df["valor"].astype(float) / 100   # % → decimal diário
        selic = df.set_index("data")["valor"]
        selic.index = selic.index.tz_localize(None)
        print(f"   SELIC: {len(selic)} dias, média anual ≈ {selic.mean()*252*100:.2f}%")
        return selic
    except Exception as e:
        print(f"   ⚠️ Erro BCB: {e}. Usando CDI = 0.")
        return pd.Series(dtype=float)

# ─── 4. CALCULAR RETORNOS DIÁRIOS ─────────────────────────────────────────────
def calc_returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change()

# ─── 5. CÁLCULO DE AR E CAR PARA UM EVENTO ────────────────────────────────────
def compute_event(
    event_date: pd.Timestamp,
    ticker: str,
    returns: pd.DataFrame,
    selic: pd.Series,
) -> dict | None:
    """
    Implementa o Market Model conforme seção 3.4 do TCC:
      (Ri - Rf) = alpha + beta*(Rm - Rf) + epsilon

    Retorna dict com ar_t0, ar_t1, car ou None se dados insuficientes.
    """
    trading_days = returns.index

    # Encontrar T0: primeiro dia útil ≥ data de publicação
    t0_pos = trading_days.searchsorted(event_date)
    if t0_pos >= len(trading_days):
        return None

    t0 = trading_days[t0_pos]
    if t0_pos + 1 >= len(trading_days):
        return None
    t1 = trading_days[t0_pos + 1]

    # Janela de estimação: [T0-120, T0-31] em dias úteis
    est_start_pos = t0_pos + ESTIMATION_START   # t0_pos - 120
    est_end_pos   = t0_pos + ESTIMATION_END + 1 # t0_pos - 30 (exclusive)

    if est_start_pos < 0:
        return None

    est_idx = trading_days[est_start_pos:est_end_pos]
    if len(est_idx) < MIN_OBS:
        return None

    # Retornos na janela de estimação
    ri_est = returns.loc[est_idx, ticker]
    rm_est = returns.loc[est_idx, IBOV]
    rf_est = selic.reindex(est_idx).ffill().fillna(0)

    ri_exc = ri_est - rf_est
    rm_exc = rm_est - rf_est

    mask = ri_exc.notna() & rm_exc.notna()
    if mask.sum() < MIN_OBS:
        return None

    # OLS: (Ri - Rf) = alpha + beta*(Rm - Rf)
    X = sm.add_constant(rm_exc[mask].values)
    y = ri_exc[mask].values
    try:
        ols  = sm.OLS(y, X).fit()
        alpha_hat = ols.params[0]
        beta_hat  = ols.params[1]
    except Exception:
        return None

    # Calcular AR para T0 e T+1
    ars = {}
    for day_label, day in [("t0", t0), ("t1", t1)]:
        if day not in returns.index:
            continue
        ri = returns.loc[day, ticker]
        rm = returns.loc[day, IBOV]
        rf = selic.get(day, 0)
        if pd.isna(ri) or pd.isna(rm):
            continue
        # E[Ri] = alpha + Rf + beta*(Rm - Rf)
        e_ri = alpha_hat + rf + beta_hat * (rm - rf)
        ars[day_label] = float(ri) - float(e_ri)

    if not ars:
        return None

    ar_t0 = ars.get("t0", np.nan)
    ar_t1 = ars.get("t1", np.nan)
    car   = sum(v for v in [ar_t0, ar_t1] if not np.isnan(v))

    return {
        "t0"   : t0,
        "ar_t0": ar_t0,
        "ar_t1": ar_t1,
        "car"  : car,
    }

# ─── 6. LOOP PRINCIPAL ────────────────────────────────────────────────────────
def run_event_study(sent_df: pd.DataFrame, returns: pd.DataFrame, selic: pd.Series) -> pd.DataFrame:
    results = []
    for _, row in tqdm(sent_df.iterrows(), total=len(sent_df), desc="Calculando AR/CAR"):
        ticker = row["ticker_yf"]
        if ticker not in returns.columns:
            continue
        ev = compute_event(row["date"], ticker, returns, selic)
        if ev is None:
            continue
        results.append({
            "idx"         : row.get("idx"),
            "ticker"      : row["ticker"],
            "date"        : row["date"].date(),
            "sentimento"  : row["sentimento"],
            "sentimento_num": row["sentimento_num"],
            **ev,
        })

    df = pd.DataFrame(results)
    df.to_csv("event_study_results.csv", index=False)
    print(f"\n✅ Eventos com CAR calculado: {len(df):,} / {len(sent_df):,}")
    print(f"   (descartados: {len(sent_df)-len(df):,} — dados insuficientes na janela de estimação)")
    return df

# ─── 7. ANÁLISE ESTATÍSTICA ───────────────────────────────────────────────────
def analyze(results_df: pd.DataFrame) -> pd.DataFrame:
    print("\n" + "="*60)
    print("ANÁLISE ESTATÍSTICA — CAR[T0, T+1]")
    print("="*60)

    table_rows = []
    groups = {}

    for sent in ["POSITIVO", "NEUTRO", "NEGATIVO"]:
        car = results_df[results_df["sentimento"] == sent]["car"].dropna()
        groups[sent] = car
        n = len(car)
        mean = car.mean()
        std  = car.std()
        t_stat, p_val = stats.ttest_1samp(car, 0)
        ci_lo, ci_hi  = stats.t.interval(0.95, n - 1, loc=mean, scale=stats.sem(car))

        print(f"\n{sent}  (N={n:,})")
        print(f"  CAR médio  : {mean*100:+.4f}%")
        print(f"  Desvio pad.: {std*100:.4f}%")
        print(f"  t-stat     : {t_stat:.4f}")
        print(f"  p-valor    : {p_val:.4f}  {'***' if p_val<0.01 else '**' if p_val<0.05 else '*' if p_val<0.10 else ''}")
        print(f"  IC 95%     : [{ci_lo*100:+.4f}%, {ci_hi*100:+.4f}%]")

        table_rows.append({
            "Sentimento": sent, "N": n,
            "CAR médio (%)": round(mean * 100, 4),
            "Desvio padrão (%)": round(std * 100, 4),
            "t-estatística": round(t_stat, 4),
            "p-valor": round(p_val, 4),
            "IC 95% inferior (%)": round(ci_lo * 100, 4),
            "IC 95% superior (%)": round(ci_hi * 100, 4),
        })

    # Teste diferença POS > NEG
    t2, p2 = stats.ttest_ind(groups["POSITIVO"], groups["NEGATIVO"], alternative="greater")
    print(f"\nTeste unilateral CAR(POSITIVO) > CAR(NEGATIVO):")
    print(f"  t = {t2:.4f}, p = {p2:.4f}  {'***' if p2<0.01 else '**' if p2<0.05 else '*' if p2<0.10 else ''}")

    # Análise por empresa
    print("\n" + "="*60)
    print("CAR MÉDIO POR EMPRESA E SENTIMENTO")
    print("="*60)
    by_company = (
        results_df.groupby(["ticker", "sentimento"])["car"]
        .agg(["count", "mean"])
        .reset_index()
    )
    by_company["mean_%"] = by_company["mean"] * 100
    by_company = by_company.sort_values(["ticker", "sentimento"])
    print(by_company.to_string(index=False))

    table = pd.DataFrame(table_rows)
    print("\n" + "="*60)
    print("TABELA RESUMO (copiar para o TCC):")
    print("="*60)
    print(table.to_string(index=False))
    table.to_csv("tabela_resultados.csv", index=False)
    by_company.to_csv("tabela_por_empresa.csv", index=False)
    print("\nArquivos salvos: tabela_resultados.csv, tabela_por_empresa.csv")
    return table

# ─── 8. GRÁFICOS ──────────────────────────────────────────────────────────────
def plot_results(results_df: pd.DataFrame):
    palette = {"POSITIVO": "#2ecc71", "NEUTRO": "#95a5a6", "NEGATIVO": "#e74c3c"}
    order   = ["POSITIVO", "NEUTRO", "NEGATIVO"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Estudo de Eventos: CAR[T₀, T₊₁] por Sentimento\n(Llama 3.1 8B | B3 2023–2024)", fontsize=12)

    # (a) Boxplot
    ax = axes[0]
    data_box = [results_df[results_df["sentimento"] == s]["car"] * 100 for s in order]
    bp = ax.boxplot(data_box, labels=order, patch_artist=True, showfliers=False)
    for patch, sent in zip(bp["boxes"], order):
        patch.set_facecolor(palette[sent])
        patch.set_alpha(0.7)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("(a) Distribuição do CAR")
    ax.set_ylabel("CAR [T₀, T₊₁] (%)")
    ax.set_xlabel("Sentimento")

    # (b) Barras com IC 95%
    ax = axes[1]
    means, errs, ns = [], [], []
    for sent in order:
        car = results_df[results_df["sentimento"] == sent]["car"] * 100
        means.append(car.mean())
        errs.append(stats.sem(car) * stats.t.ppf(0.975, len(car) - 1))
        ns.append(len(car))
    bars = ax.bar(order, means, color=[palette[s] for s in order], alpha=0.8, width=0.5)
    ax.errorbar(order, means, yerr=errs, fmt="none", color="black", capsize=5, linewidth=1.5)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("(b) CAR Médio (IC 95%)")
    ax.set_ylabel("CAR Médio (%)")
    for bar, n in zip(bars, ns):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"n={n:,}", ha="center", va="bottom", fontsize=8)

    # (c) CAR médio por empresa (apenas POS e NEG)
    ax = axes[2]
    pivot = (
        results_df[results_df["sentimento"].isin(["POSITIVO", "NEGATIVO"])]
        .groupby(["ticker", "sentimento"])["car"]
        .mean()
        .unstack() * 100
    )
    pivot.plot(kind="bar", ax=ax, color=[palette["POSITIVO"], palette["NEGATIVO"]],
               alpha=0.8, width=0.6)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("(c) CAR Médio por Empresa")
    ax.set_xlabel("")
    ax.set_ylabel("CAR Médio (%)")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.legend(title="Sentimento", fontsize=8)

    plt.tight_layout()
    plt.savefig("graficos_car.png", dpi=180, bbox_inches="tight")
    print("📊 Gráfico salvo: graficos_car.png")

    # Gráfico adicional: distribuição de sentimentos
    fig2, ax2 = plt.subplots(figsize=(8, 4))
    sent_counts = results_df["sentimento"].value_counts()[order]
    bars = ax2.bar(order, sent_counts.values, color=[palette[s] for s in order], alpha=0.8, width=0.5)
    for bar, v in zip(bars, sent_counts.values):
        ax2.text(bar.get_x() + bar.get_width()/2, v + 5, str(v),
                 ha="center", va="bottom", fontsize=10)
    ax2.set_title("Distribuição de Sentimentos Classificados pelo Llama 3.1 8B")
    ax2.set_ylabel("Número de Manchetes")
    ax2.set_xlabel("Sentimento")
    plt.tight_layout()
    plt.savefig("distribuicao_sentimentos.png", dpi=180, bbox_inches="tight")
    print("📊 Gráfico salvo: distribuicao_sentimentos.png")

# ─── MAIN ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Verificações
    if not Path("sentimentos.jsonl").exists():
        print("❌ sentimentos.jsonl não encontrado.")
        print("   Rode primeiro: python 1_score_sentimentos.py")
        raise SystemExit(1)

    # Carregar sentimentos
    print("📖 Carregando sentimentos...")
    sent_df = load_sentimentos()
    print(f"   {len(sent_df):,} eventos carregados")
    print(f"   Distribuição:\n{sent_df['sentimento'].value_counts().to_string()}")

    # Download de dados financeiros
    prices  = download_prices()
    returns = calc_returns(prices)
    selic   = download_selic()

    # Event study
    print("\n📐 Rodando estudo de eventos...")
    results_df = run_event_study(sent_df, returns, selic)

    # Análise
    analyze(results_df)
    plot_results(results_df)

    print("\n✅ Tudo pronto!")
    print("   Arquivos gerados:")
    print("   - event_study_results.csv  (dados completos por evento)")
    print("   - tabela_resultados.csv    (tabela para o TCC)")
    print("   - tabela_por_empresa.csv   (breakdown por empresa)")
    print("   - graficos_car.png         (gráfico principal)")
    print("   - distribuicao_sentimentos.png")

In [ ]:
"""
SCRIPT 2: Estudo de Eventos – Pipeline Completo
===============================================
Uso (após 1_score_sentimentos.py ter terminado):
    pip install pandas yfinance scipy statsmodels requests matplotlib seaborn tqdm
    python 2_event_study.py

Requer: sentimentos.jsonl (gerado pelo script 1)
        dataset_10_gigantes_tcc.csv (base de notícias original)
Output: event_study_results.csv, tabelas no terminal, gráficos .png
"""

import json
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm
import yfinance as yf
from scipy import stats
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ─── PARÂMETROS (conforme metodologia do TCC) ─────────────────────────────────
ESTIMATION_START = -120   # dias úteis antes de T0 (início da janela de estimação)
ESTIMATION_END   = -31    # dias úteis antes de T0 (fim da janela de estimação)
# Janela de evento: T0 e T+1 → CAR[0,+1]
MIN_OBS = 30              # mínimo de observações na janela de estimação
# ──────────────────────────────────────────────────────────────────────────────

TICKERS = [
    "PETR4.SA", "BBDC4.SA", "BBAS3.SA", "ABEV3.SA",
    "AXIA3.SA", "BPAC11.SA", "ITUB4.SA", "VALE3.SA",
    "WEGE3.SA", "ITSA4.SA",
]
IBOV = "^BVSP"
# Mapeamento ticker do CSV → ticker yfinance
# ELET3 mudou de nome para AXIA3 no fim de 2025; o histórico de preços na
# Yahoo Finance ficou associado ao novo símbolo, então mapeamos manualmente.
TICKER_MAP = {t.replace(".SA", ""): t for t in TICKERS}
TICKER_MAP["ELET3"] = "AXIA3.SA"

# ─── 1. CARREGAR SENTIMENTOS ───────────────────────────────────────────────────
def load_sentimentos(path="sentimentos.jsonl") -> pd.DataFrame:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get("sentimento") not in ("ERRO", None):
                    records.append(r)
            except Exception:
                pass
    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["datetime"])
    df["ticker_yf"] = df["ticker"].map(TICKER_MAP)
    df = df.dropna(subset=["ticker_yf"])
    mapping = {"POSITIVO": 1, "NEUTRO": 0, "NEGATIVO": -1}
    df["sentimento_num"] = df["sentimento"].map(mapping)
    return df

# ─── 2. DOWNLOAD DE PREÇOS (yfinance) ─────────────────────────────────────────
def download_prices() -> pd.DataFrame:
    print("📥 Baixando preços (yfinance)...")
    raw = yf.download(
        TICKERS + [IBOV],
        start="2022-06-01",
        end="2025-01-31",
        auto_adjust=True,
        progress=True,
    )
    # yfinance pode retornar MultiIndex ou Index simples dependendo da versão
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw["Close"]
    else:
        prices = raw[["Close"]] if "Close" in raw.columns else raw
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    prices = prices.dropna(how="all")
    print(f"   Preços: {prices.shape[0]} dias úteis, {prices.shape[1]} ativos")
    return prices

# ─── 3. DOWNLOAD SELIC/CDI (BCB SGS) ─────────────────────────────────────────
def download_selic() -> pd.Series:
    """Taxa CDI diária (série 11 do BCB), já em decimal (ex: 0.0004)."""
    print("📥 Baixando Taxa CDI/SELIC (BCB)...")
    url = (
        "https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados"
        "?formato=json&dataInicial=01/06/2022&dataFinal=31/01/2025"
    )
    try:
        r = requests.get(url, timeout=30)
        data = r.json()
        df = pd.DataFrame(data)
        df["data"]  = pd.to_datetime(df["data"], format="%d/%m/%Y")
        df["valor"] = df["valor"].astype(float) / 100   # % → decimal diário
        selic = df.set_index("data")["valor"]
        selic.index = selic.index.tz_localize(None)
        print(f"   SELIC: {len(selic)} dias, média anual ≈ {selic.mean()*252*100:.2f}%")
        return selic
    except Exception as e:
        print(f"   ⚠️ Erro BCB: {e}. Usando CDI = 0.")
        return pd.Series(dtype=float)

# ─── 4. CALCULAR RETORNOS DIÁRIOS ─────────────────────────────────────────────
def calc_returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change()

# ─── 5. CÁLCULO DE AR E CAR PARA UM EVENTO ────────────────────────────────────
def compute_event(
    event_date: pd.Timestamp,
    ticker: str,
    returns: pd.DataFrame,
    selic: pd.Series,
) -> dict | None:
    """
    Implementa o Market Model conforme seção 3.4 do TCC:
      (Ri - Rf) = alpha + beta*(Rm - Rf) + epsilon

    Retorna dict com ar_t0, ar_t1, car ou None se dados insuficientes.
    """
    trading_days = returns.index

    # Encontrar T0: primeiro dia útil ≥ data de publicação
    t0_pos = trading_days.searchsorted(event_date)
    if t0_pos >= len(trading_days):
        return None

    t0 = trading_days[t0_pos]
    if t0_pos + 1 >= len(trading_days):
        return None
    t1 = trading_days[t0_pos + 1]

    # Janela de estimação: [T0-120, T0-31] em dias úteis
    est_start_pos = t0_pos + ESTIMATION_START   # t0_pos - 120
    est_end_pos   = t0_pos + ESTIMATION_END + 1 # t0_pos - 30 (exclusive)

    if est_start_pos < 0:
        return None

    est_idx = trading_days[est_start_pos:est_end_pos]
    if len(est_idx) < MIN_OBS:
        return None

    # Retornos na janela de estimação
    ri_est = returns.loc[est_idx, ticker]
    rm_est = returns.loc[est_idx, IBOV]
    rf_est = selic.reindex(est_idx).ffill().fillna(0)

    ri_exc = ri_est - rf_est
    rm_exc = rm_est - rf_est

    mask = ri_exc.notna() & rm_exc.notna()
    if mask.sum() < MIN_OBS:
        return None

    # OLS: (Ri - Rf) = alpha + beta*(Rm - Rf)
    X = sm.add_constant(rm_exc[mask].values)
    y = ri_exc[mask].values
    try:
        ols  = sm.OLS(y, X).fit()
        alpha_hat = ols.params[0]
        beta_hat  = ols.params[1]
    except Exception:
        return None

    # Calcular AR para T0 e T+1
    ars = {}
    for day_label, day in [("t0", t0), ("t1", t1)]:
        if day not in returns.index:
            continue
        ri = returns.loc[day, ticker]
        rm = returns.loc[day, IBOV]
        rf = selic.get(day, 0)
        if pd.isna(ri) or pd.isna(rm):
            continue
        # E[Ri] = alpha + Rf + beta*(Rm - Rf)
        e_ri = alpha_hat + rf + beta_hat * (rm - rf)
        ars[day_label] = float(ri) - float(e_ri)

    if not ars:
        return None

    ar_t0 = ars.get("t0", np.nan)
    ar_t1 = ars.get("t1", np.nan)
    car   = sum(v for v in [ar_t0, ar_t1] if not np.isnan(v))

    return {
        "t0"   : t0,
        "ar_t0": ar_t0,
        "ar_t1": ar_t1,
        "car"  : car,
    }

# ─── 6. LOOP PRINCIPAL ────────────────────────────────────────────────────────
def run_event_study(sent_df: pd.DataFrame, returns: pd.DataFrame, selic: pd.Series) -> pd.DataFrame:
    results = []
    for _, row in tqdm(sent_df.iterrows(), total=len(sent_df), desc="Calculando AR/CAR"):
        ticker = row["ticker_yf"]
        if ticker not in returns.columns:
            continue
        ev = compute_event(row["date"], ticker, returns, selic)
        if ev is None:
            continue
        results.append({
            "idx"         : row.get("idx"),
            "ticker"      : row["ticker"],
            "date"        : row["date"].date(),
            "sentimento"  : row["sentimento"],
            "sentimento_num": row["sentimento_num"],
            **ev,
        })

    df = pd.DataFrame(results)
    df.to_csv("event_study_results.csv", index=False)
    print(f"\n✅ Eventos com CAR calculado: {len(df):,} / {len(sent_df):,}")
    print(f"   (descartados: {len(sent_df)-len(df):,} — dados insuficientes na janela de estimação)")
    return df

# ─── 7. ANÁLISE ESTATÍSTICA ───────────────────────────────────────────────────
def analyze(results_df: pd.DataFrame) -> pd.DataFrame:
    print("\n" + "="*60)
    print("ANÁLISE ESTATÍSTICA — CAR[T0, T+1]")
    print("="*60)

    table_rows = []
    groups = {}

    for sent in ["POSITIVO", "NEUTRO", "NEGATIVO"]:
        car = results_df[results_df["sentimento"] == sent]["car"].dropna()
        groups[sent] = car
        n = len(car)
        mean = car.mean()
        std  = car.std()
        t_stat, p_val = stats.ttest_1samp(car, 0)
        ci_lo, ci_hi  = stats.t.interval(0.95, n - 1, loc=mean, scale=stats.sem(car))

        print(f"\n{sent}  (N={n:,})")
        print(f"  CAR médio  : {mean*100:+.4f}%")
        print(f"  Desvio pad.: {std*100:.4f}%")
        print(f"  t-stat     : {t_stat:.4f}")
        print(f"  p-valor    : {p_val:.4f}  {'***' if p_val<0.01 else '**' if p_val<0.05 else '*' if p_val<0.10 else ''}")
        print(f"  IC 95%     : [{ci_lo*100:+.4f}%, {ci_hi*100:+.4f}%]")

        table_rows.append({
            "Sentimento": sent, "N": n,
            "CAR médio (%)": round(mean * 100, 4),
            "Desvio padrão (%)": round(std * 100, 4),
            "t-estatística": round(t_stat, 4),
            "p-valor": round(p_val, 4),
            "IC 95% inferior (%)": round(ci_lo * 100, 4),
            "IC 95% superior (%)": round(ci_hi * 100, 4),
        })

    # Teste diferença POS > NEG
    t2, p2 = stats.ttest_ind(groups["POSITIVO"], groups["NEGATIVO"], alternative="greater")
    print(f"\nTeste unilateral CAR(POSITIVO) > CAR(NEGATIVO):")
    print(f"  t = {t2:.4f}, p = {p2:.4f}  {'***' if p2<0.01 else '**' if p2<0.05 else '*' if p2<0.10 else ''}")

    # Análise por empresa
    print("\n" + "="*60)
    print("CAR MÉDIO POR EMPRESA E SENTIMENTO")
    print("="*60)
    by_company = (
        results_df.groupby(["ticker", "sentimento"])["car"]
        .agg(["count", "mean"])
        .reset_index()
    )
    by_company["mean_%"] = by_company["mean"] * 100
    by_company = by_company.sort_values(["ticker", "sentimento"])
    print(by_company.to_string(index=False))

    table = pd.DataFrame(table_rows)
    print("\n" + "="*60)
    print("TABELA RESUMO (copiar para o TCC):")
    print("="*60)
    print(table.to_string(index=False))
    table.to_csv("tabela_resultados.csv", index=False)
    by_company.to_csv("tabela_por_empresa.csv", index=False)
    print("\nArquivos salvos: tabela_resultados.csv, tabela_por_empresa.csv")
    return table

# ─── 8. GRÁFICOS ──────────────────────────────────────────────────────────────
def plot_results(results_df: pd.DataFrame):
    palette = {"POSITIVO": "#2ecc71", "NEUTRO": "#95a5a6", "NEGATIVO": "#e74c3c"}
    order   = ["POSITIVO", "NEUTRO", "NEGATIVO"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Estudo de Eventos: CAR[T₀, T₊₁] por Sentimento\n(Llama 3.1 8B | B3 2023–2024)", fontsize=12)

    # (a) Boxplot
    ax = axes[0]
    data_box = [results_df[results_df["sentimento"] == s]["car"] * 100 for s in order]
    bp = ax.boxplot(data_box, labels=order, patch_artist=True, showfliers=False)
    for patch, sent in zip(bp["boxes"], order):
        patch.set_facecolor(palette[sent])
        patch.set_alpha(0.7)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("(a) Distribuição do CAR")
    ax.set_ylabel("CAR [T₀, T₊₁] (%)")
    ax.set_xlabel("Sentimento")

    # (b) Barras com IC 95%
    ax = axes[1]
    means, errs, ns = [], [], []
    for sent in order:
        car = results_df[results_df["sentimento"] == sent]["car"] * 100
        means.append(car.mean())
        errs.append(stats.sem(car) * stats.t.ppf(0.975, len(car) - 1))
        ns.append(len(car))
    bars = ax.bar(order, means, color=[palette[s] for s in order], alpha=0.8, width=0.5)
    ax.errorbar(order, means, yerr=errs, fmt="none", color="black", capsize=5, linewidth=1.5)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("(b) CAR Médio (IC 95%)")
    ax.set_ylabel("CAR Médio (%)")
    for bar, n in zip(bars, ns):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"n={n:,}", ha="center", va="bottom", fontsize=8)

    # (c) CAR médio por empresa (apenas POS e NEG)
    ax = axes[2]
    pivot = (
        results_df[results_df["sentimento"].isin(["POSITIVO", "NEGATIVO"])]
        .groupby(["ticker", "sentimento"])["car"]
        .mean()
        .unstack() * 100
    )
    pivot.plot(kind="bar", ax=ax, color=[palette["POSITIVO"], palette["NEGATIVO"]],
               alpha=0.8, width=0.6)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("(c) CAR Médio por Empresa")
    ax.set_xlabel("")
    ax.set_ylabel("CAR Médio (%)")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.legend(title="Sentimento", fontsize=8)

    plt.tight_layout()
    plt.savefig("graficos_car.png", dpi=180, bbox_inches="tight")
    print("📊 Gráfico salvo: graficos_car.png")

    # Gráfico adicional: distribuição de sentimentos
    fig2, ax2 = plt.subplots(figsize=(8, 4))
    sent_counts = results_df["sentimento"].value_counts()[order]
    bars = ax2.bar(order, sent_counts.values, color=[palette[s] for s in order], alpha=0.8, width=0.5)
    for bar, v in zip(bars, sent_counts.values):
        ax2.text(bar.get_x() + bar.get_width()/2, v + 5, str(v),
                 ha="center", va="bottom", fontsize=10)
    ax2.set_title("Distribuição de Sentimentos Classificados pelo Llama 3.1 8B")
    ax2.set_ylabel("Número de Manchetes")
    ax2.set_xlabel("Sentimento")
    plt.tight_layout()
    plt.savefig("distribuicao_sentimentos.png", dpi=180, bbox_inches="tight")
    print("📊 Gráfico salvo: distribuicao_sentimentos.png")

# ─── MAIN ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Verificações
    if not Path("sentimentos.jsonl").exists():
        print("❌ sentimentos.jsonl não encontrado.")
        print("   Rode primeiro: python 1_score_sentimentos.py")
        raise SystemExit(1)

    # Carregar sentimentos
    print("📖 Carregando sentimentos...")
    sent_df = load_sentimentos()
    print(f"   {len(sent_df):,} eventos carregados")
    print(f"   Distribuição:\n{sent_df['sentimento'].value_counts().to_string()}")

    # Download de dados financeiros
    prices  = download_prices()
    returns = calc_returns(prices)
    selic   = download_selic()

    # Event study
    print("\n📐 Rodando estudo de eventos...")
    results_df = run_event_study(sent_df, returns, selic)

    # Análise
    analyze(results_df)
    plot_results(results_df)

    print("\n✅ Tudo pronto!")
    print("   Arquivos gerados:")
    print("   - event_study_results.csv  (dados completos por evento)")
    print("   - tabela_resultados.csv    (tabela para o TCC)")
    print("   - tabela_por_empresa.csv   (breakdown por empresa)")
    print("   - graficos_car.png         (gráfico principal)")
    print("   - distribuicao_sentimentos.png")

In [ ]:
pip install pandas numpy scipy statsmodels yfinance requests matplotlib


In [ ]:
"""
SCRIPT 3: Acurácia Direcional + Regressão Logística + Portfólio Long/Short
============================================================================
Uso:
    pip install pandas numpy scipy statsmodels yfinance requests matplotlib
    python 3_logistic_e_portfolio.py

Requer: event_study_results.csv (gerado pelo script 2)
Output: tabela_acuracia.csv, tabela_logit.csv, portfolio_resultados.csv,
        grafico_portfolio.png
"""

import warnings
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm
import statsmodels.formula.api as smf
import yfinance as yf
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RESULTS_CSV = "event_study_results.csv"

TICKERS = [
    "PETR4.SA", "BBDC4.SA", "BBAS3.SA", "ABEV3.SA",
    "AXIA3.SA", "BPAC11.SA", "ITUB4.SA", "VALE3.SA",
    "WEGE3.SA", "ITSA4.SA",
]
IBOV = "^BVSP"
TICKER_MAP = {t.replace(".SA", ""): t for t in TICKERS}
TICKER_MAP["ELET3"] = "AXIA3.SA"

# ═══════════════════════════════════════════════════════════════════════
# PARTE 1 — ACURÁCIA DIRECIONAL E REGRESSÃO LOGÍSTICA (seção 4.3)
# ═══════════════════════════════════════════════════════════════════════

def secao_4_3(df: pd.DataFrame):
    print("=" * 65)
    print("4.3 — ACURÁCIA DIRECIONAL E REGRESSÃO LOGÍSTICA")
    print("=" * 65)

    # Descarta NEUTRO: não é previsão direcional (decisão metodológica,
    # ver Lopez-Lira & Tang, 2023)
    df2 = df[df["sentimento"] != "NEUTRO"].copy()
    df2["y_real"] = (df2["car"] > 0).astype(int)
    df2["y_pred"] = (df2["sentimento"] == "POSITIVO").astype(int)
    print(f"\nN (excluindo NEUTRO): {len(df2):,}")

    # ── Baseline justo: % de dias em que o retorno simplesmente foi positivo
    baseline_acc = max(df2["y_real"].mean(), 1 - df2["y_real"].mean())
    modelo_acc = (df2["y_real"] == df2["y_pred"]).mean()

    print(f"\nBaseline (sempre prever a classe majoritária): {baseline_acc*100:.2f}%")
    print(f"Acurácia do LLM (sentimento → direção):          {modelo_acc*100:.2f}%")

    # ── Teste de McNemar (compara 2 classificadores na mesma amostra)
    pred_baseline = np.full(len(df2), int(df2["y_real"].mean() > 0.5))
    acerto_llm = (df2["y_pred"] == df2["y_real"]).astype(int)
    acerto_base = (pred_baseline == df2["y_real"]).astype(int)

    # tabela 2x2: LLM certo/errado vs baseline certo/errado
    n_ambos_certos   = ((acerto_llm == 1) & (acerto_base == 1)).sum()
    n_so_llm_certo   = ((acerto_llm == 1) & (acerto_base == 0)).sum()
    n_so_base_certo  = ((acerto_llm == 0) & (acerto_base == 1)).sum()
    n_ambos_errados  = ((acerto_llm == 0) & (acerto_base == 0)).sum()

    b, c = n_so_llm_certo, n_so_base_certo
    if b + c > 0:
        mcnemar_stat = (abs(b - c) - 1) ** 2 / (b + c)
        mcnemar_p = 1 - stats.chi2.cdf(mcnemar_stat, df=1)
    else:
        mcnemar_stat, mcnemar_p = np.nan, np.nan

    print(f"\nTeste de McNemar (LLM vs. baseline majoritário):")
    print(f"  estatística = {mcnemar_stat:.4f}, p-valor = {mcnemar_p:.4f}")
    sig = "***" if mcnemar_p < 0.01 else "**" if mcnemar_p < 0.05 else "*" if mcnemar_p < 0.10 else "n.s."
    print(f"  {sig}")

    # ── Matriz de confusão
    print("\nMatriz de confusão:")
    print(f"                    Real: CAR>0   Real: CAR<=0")
    print(f"  Previsto Positivo:   {((df2['y_pred']==1)&(df2['y_real']==1)).sum():>6}        {((df2['y_pred']==1)&(df2['y_real']==0)).sum():>6}")
    print(f"  Previsto Negativo:   {((df2['y_pred']==0)&(df2['y_real']==1)).sum():>6}        {((df2['y_pred']==0)&(df2['y_real']==0)).sum():>6}")

    # ── Regressão logística com efeitos fixos por empresa
    print("\n" + "-" * 65)
    print("Regressão Logística: P(CAR>0) ~ Sentimento + Efeitos Fixos(Empresa)")
    print("-" * 65)

    modelo = smf.logit("y_real ~ sentimento_num + C(ticker)", data=df2).fit(disp=0)
    print(modelo.summary())

    # Odds ratio do coeficiente de sentimento
    coef_sent = modelo.params["sentimento_num"]
    odds_ratio = np.exp(coef_sent)
    p_valor_sent = modelo.pvalues["sentimento_num"]
    print(f"\nOdds Ratio (sentimento): {odds_ratio:.4f}")
    print(f"Interpretação: notícia positiva multiplica por {odds_ratio:.2f}x a chance de")
    print(f"retorno anormal positivo, controlando por empresa (p={p_valor_sent:.4f}).")
    print(f"Pseudo R² (McFadden): {modelo.prsquared:.4f}")

    # Salva tabela resumo
    tabela = pd.DataFrame([{
        "N": len(df2),
        "Acurácia LLM (%)": round(modelo_acc * 100, 2),
        "Baseline (%)": round(baseline_acc * 100, 2),
        "McNemar p-valor": round(mcnemar_p, 4),
        "Odds Ratio (sentimento)": round(odds_ratio, 4),
        "P-valor (sentimento)": round(p_valor_sent, 4),
        "Pseudo R²": round(modelo.prsquared, 4),
    }])
    tabela.to_csv("tabela_acuracia.csv", index=False)
    with open("tabela_logit_summary.txt", "w", encoding="utf-8") as f:
        f.write(str(modelo.summary()))
    print("\nArquivos salvos: tabela_acuracia.csv, tabela_logit_summary.txt")

    return df2


# ═══════════════════════════════════════════════════════════════════════
# PARTE 2 — PORTFÓLIO LONG/SHORT E SHARPE RATIO (seção 4.4)
# ═══════════════════════════════════════════════════════════════════════

def download_prices():
    print("\n📥 Baixando preços para o backtest...")
    raw = yf.download(TICKERS + [IBOV], start="2022-06-01", end="2025-01-31",
                       auto_adjust=True, progress=False)
    prices = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    return prices.dropna(how="all")

def download_selic():
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados"
           "?formato=json&dataInicial=01/06/2022&dataFinal=31/01/2025")
    try:
        r = requests.get(url, timeout=30)
        df = pd.DataFrame(r.json())
        df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
        df["valor"] = df["valor"].astype(float) / 100
        s = df.set_index("data")["valor"]
        s.index = s.index.tz_localize(None)
        return s
    except Exception:
        return pd.Series(dtype=float)


def secao_4_4(df: pd.DataFrame):
    print("\n" + "=" * 65)
    print("4.4 — BACKTEST DE PORTFÓLIO LONG/SHORT")
    print("=" * 65)
    print("""
NOTA METODOLÓGICA (importante para o texto do TCC):
O retorno do portfólio em cada dia de sinal é aproximado pelo CAR[T0,T+1]
já calculado no estudo de eventos — ou seja, assume-se que a posição é
aberta no dia do sinal e o retorno de 2 dias é integralmente atribuído
a esse dia (simplificação padrão em backtests baseados em eventos).
Múltiplas notícias da mesma empresa no mesmo dia são agregadas pela
sentimento médio (sentimento_num médio) antes de definir a posição.
""")

    df = df[df["sentimento"] != "NEUTRO"].copy()
    df["t0"] = pd.to_datetime(df["t0"])

    # Agrega múltiplos sinais da mesma empresa no mesmo dia
    daily = (
        df.groupby(["t0", "ticker"])
        .agg(sentimento_medio=("sentimento_num", "mean"), car=("car", "mean"))
        .reset_index()
    )
    daily["posicao"] = np.sign(daily["sentimento_medio"])
    daily = daily[daily["posicao"] != 0]

    # Para cada dia, separa longs e shorts, calcula retorno do portfólio
    portfolio = []
    for date, grupo in daily.groupby("t0"):
        longs = grupo[grupo["posicao"] == 1]
        shorts = grupo[grupo["posicao"] == -1]
        if len(longs) == 0 and len(shorts) == 0:
            continue
        r_long = longs["car"].mean() if len(longs) > 0 else 0
        r_short = shorts["car"].mean() if len(shorts) > 0 else 0
        # Long/short dólar-neutro: ganha com a diferença
        r_portfolio = (r_long - r_short) / 2
        portfolio.append({
            "date": date, "n_long": len(longs), "n_short": len(shorts),
            "r_long": r_long, "r_short": r_short, "r_portfolio": r_portfolio,
        })

    port_df = pd.DataFrame(portfolio).sort_values("date").reset_index(drop=True)
    port_df.to_csv("portfolio_resultados.csv", index=False)
    print(f"Dias de portfólio com posição: {len(port_df)}")

    # ── Sharpe Ratio do portfólio long/short
    def sharpe_anualizado(retornos, rf_diario=0):
        excess = retornos - rf_diario
        sharpe_diario = excess.mean() / excess.std()
        return sharpe_diario * np.sqrt(252)

    def erro_padrao_sharpe_lo(sharpe_diario, n):
        # Aproximação IID de Lo (2002) para o erro-padrão do Sharpe estimado
        return np.sqrt((1 + 0.5 * sharpe_diario ** 2) / n)

    sharpe_port = sharpe_anualizado(port_df["r_portfolio"])
    n_port = len(port_df)
    sharpe_diario_port = port_df["r_portfolio"].mean() / port_df["r_portfolio"].std()
    se_port = erro_padrao_sharpe_lo(sharpe_diario_port, n_port) * np.sqrt(252)

    print(f"\nPortfólio Long/Short (sinal LLM):")
    print(f"  Retorno médio diário: {port_df['r_portfolio'].mean()*100:.4f}%")
    print(f"  Sharpe anualizado:    {sharpe_port:.4f}  (erro-padrão ≈ {se_port:.4f})")
    print(f"  IC 95% aprox.:        [{sharpe_port-1.96*se_port:.4f}, {sharpe_port+1.96*se_port:.4f}]")

    # ── Benchmarks: Ibovespa e cesta igual-ponderada das 10 ações (buy & hold)
    prices = download_prices()
    returns = prices.pct_change().dropna(how="all")

    ibov_ret = returns[IBOV].dropna()
    sharpe_ibov = sharpe_anualizado(ibov_ret)

    cesta_ret = returns[TICKERS].mean(axis=1).dropna()
    sharpe_cesta = sharpe_anualizado(cesta_ret)

    print(f"\nBenchmarks (buy & hold, mesmo período de preços):")
    print(f"  Sharpe Ibovespa:                  {sharpe_ibov:.4f}")
    print(f"  Sharpe cesta igual-ponderada (10): {sharpe_cesta:.4f}")
    print(f"  Sharpe portfólio long/short (LLM): {sharpe_port:.4f}")

    # Tabela resumo
    resumo = pd.DataFrame([
        {"Estratégia": "Long/Short (sinal LLM)", "Sharpe": round(sharpe_port, 4),
         "Erro-padrão (Lo, 2002)": round(se_port, 4), "N dias": n_port},
        {"Estratégia": "Ibovespa (buy & hold)", "Sharpe": round(sharpe_ibov, 4),
         "Erro-padrão (Lo, 2002)": np.nan, "N dias": len(ibov_ret)},
        {"Estratégia": "Cesta 10 ações (buy & hold)", "Sharpe": round(sharpe_cesta, 4),
         "Erro-padrão (Lo, 2002)": np.nan, "N dias": len(cesta_ret)},
    ])
    resumo.to_csv("tabela_sharpe.csv", index=False)
    print("\nArquivos salvos: portfolio_resultados.csv, tabela_sharpe.csv")

    # ── Gráfico: retorno acumulado
    port_df["cum_return"] = (1 + port_df["r_portfolio"]).cumprod() - 1
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(port_df["date"], port_df["cum_return"] * 100, label="Long/Short (sinal LLM)", linewidth=1.8)
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_title("Retorno Acumulado: Estratégia Long/Short Baseada em Sentimento (Llama 3.1 8B)")
    ax.set_ylabel("Retorno Acumulado (%)")
    ax.set_xlabel("Data")
    ax.legend()
    plt.tight_layout()
    plt.savefig("grafico_portfolio.png", dpi=180, bbox_inches="tight")
    print("📊 Gráfico salvo: grafico_portfolio.png")

    return port_df, resumo


if __name__ == "__main__":
    print("📖 Carregando resultados do estudo de eventos...")
    df = pd.read_csv(RESULTS_CSV)
    print(f"   {len(df):,} eventos carregados\n")

    df2 = secao_4_3(df)
    port_df, resumo = secao_4_4(df)

    print("\n" + "=" * 65)
    print("✅ ANÁLISE COMPLETA")
    print("=" * 65)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("event_study_results.csv")
df["t0"] = pd.to_datetime(df["t0"])

# Para cada dia com mais de uma empresa, calcula correlação par a par dos CARs
dias_multiplos = df.groupby("t0").filter(lambda x: len(x) > 1)

correlacoes = []
for date, grupo in dias_multiplos.groupby("t0"):
    cars = grupo["car"].values
    n = len(cars)
    if n < 2:
        continue
    # Correlações par a par
    for i in range(n):
        for j in range(i+1, n):
            correlacoes.append(cars[i] * cars[j])  # proxy de covariância

print(f"N de pares: {len(correlacoes)}")
print(f"Correlação média (proxy): {np.mean(correlacoes):.6f}")
print(f"t-stat (zero = sem dependência): {np.mean(correlacoes)/np.std(correlacoes)*np.sqrt(len(correlacoes)):.4f}")

In [ ]:
import statsmodels.formula.api as smf

# Codifica sentimento numericamente
sentimento_map = {"POSITIVO": 1, "NEUTRO": 0, "NEGATIVO": -1}
df["S"] = df["sentimento"].map(sentimento_map)

# OLS com erros clusterizados por data
modelo = smf.ols("car ~ C(sentimento)", data=df).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["t0"]}
)
print(modelo.summary())

In [ ]:
import pandas as pd
import numpy as np

# Carrega o CSV gerado pelo script 2 (event study)
# Se não estiver no ambiente, sobe via upload primeiro
df = pd.read_csv("event_study_results.csv")

# Reconstrói as colunas necessárias
df["date"] = pd.to_datetime(df["t0"])
mapping = {"POSITIVO": 1, "NEUTRO": 0, "NEGATIVO": -1}
df["sentimento_num"] = df["sentimento"].map(mapping)

# df2 = apenas POSITIVO e NEGATIVO (exclui NEUTRO)
df2 = df[df["sentimento"] != "NEUTRO"].copy()
df2["y_real"] = (df2["car"] > 0).astype(int)

print(f"N total: {len(df)}")
print(f"N sem NEUTRO (df2): {len(df2)}")
print(df2["sentimento"].value_counts())

In [ ]:
modelo = smf.logit("y_real ~ sentimento_num + C(ticker)", data=df2).fit(
    cov_type="cluster",
    cov_kwds={"groups": df2["date"]}
)

In [ ]:
# Versão simplificada do Link test para logit
fitted = modelo.fittedvalues  # log-odds preditos
df2["yhat"] = fitted
df2["yhat2"] = fitted ** 2

link_test = smf.logit("y_real ~ yhat + yhat2", data=df2).fit(disp=0)
p_yhat2 = link_test.pvalues["yhat2"]
print(f"Link test p-valor do termo quadrático: {p_yhat2:.4f}")
# Se p > 0,05 → forma funcional não rejeitada

In [ ]:
from scipy import stats

# Dividir em 10 grupos por probabilidade predita
df2["prob"] = modelo.predict()
df2["decil"] = pd.qcut(df2["prob"], 10, labels=False)

hl_stat = 0
for _, grupo in df2.groupby("decil"):
    obs = grupo["y_real"].sum()
    exp = grupo["prob"].sum()
    n = len(grupo)
    obs_neg = n - obs
    exp_neg = n - exp
    hl_stat += (obs - exp)**2 / exp + (obs_neg - exp_neg)**2 / exp_neg

p_hl = 1 - stats.chi2.cdf(hl_stat, df=8)
print(f"Hosmer-Lemeshow: χ²={hl_stat:.4f}, p={p_hl:.4f}")

In [ ]:
import statsmodels.formula.api as smf

modelo_logit = smf.logit("y_real ~ sentimento_num + C(ticker)", data=df2).fit(
    cov_type="cluster",
    cov_kwds={"groups": df2["date"]},
    disp=0
)
print(modelo_logit.summary())
print(f"\nOdds Ratio (sentimento): {np.exp(modelo_logit.params['sentimento_num']):.4f}")